# Faruq-v3 AF2 → FFAB2 → DCT — ALL SEEDS / ALL STAGES (Kaggle)

Satu notebook untuk seluruh eksperimen frozen saat ini. Urutan otomatis:

1. Stage 1: `AF2FS` vs `AF2FFAB2FS` untuk seed **42, 123, 2026**.
2. Jalankan keputusan tiga-seed Stage 1.
3. **Hanya jika Stage 1 PASS**, jalankan `AF2FFADCTFS` untuk seed **42, 123, 2026**.
4. Jalankan keputusan akurasi + efisiensi DCT vs rFFT.

Required Kaggle Input: private dataset `faruq-v3-experiment-core-v1` versi terbaru yang memuat `af2_spectral_kaggle_manifest.json`, archive development, dan D0 seed 42/123/2026.

GPU dan Internet harus ON. Test tetap terkunci. Tidak ada STB/top-controls di workflow ini.

Opsional untuk resume antar Saved Version: attach output notebook ini sebelumnya sebagai Kaggle Input. Notebook akan mencari `af2-ffab2-all-seeds-all-stages-state.zip`, restore state terakhir yang tersimpan, lalu skip/resume run sesuai kontrak.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
required_names=(
    'af2_spectral_kaggle_manifest.json',
    'faruq-development-v3-grouped.tar.bin',
    'D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt',
)
for name in required_names:
    matches=sorted(INPUT.rglob(name))
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    print('INPUT OK:',name,'->',matches[0])
print('KAGGLE INPUT PREFLIGHT PASS')

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,torch,zipfile
BRANCH='codex/af2-ffab2-from-start-dct'
REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH); print('COMMIT:',COMMIT); print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')

In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CORE_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
if CORE_CONTRACT.get('decision')!='PASS' or CORE_CONTRACT.get('test_images_accessed') is not False:
    raise RuntimeError('Core Kaggle tidak lolos kontrak/test-lock.')
GROUPED=DATA/'faruq_grouped_summary.json'
if not GROUPED.is_file() or (DATA/'test').exists(): raise RuntimeError('Grouped development/test-lock tidak valid.')
SEEDS=(42,123,2026)
D0={seed:Path(ARTIFACTS[f'D0_seed{seed}_best.pt']) for seed in SEEDS}
OUT=WORK/'af2-ffab2-all-seeds-all-stages-v1'
STATE_ZIP=WORK/'af2-ffab2-all-seeds-all-stages-state.zip'
prior=sorted(INPUT.rglob(STATE_ZIP.name))
if len(prior)>1: raise RuntimeError(f'Resume state ambigu: {prior}')
if len(prior)==1 and not OUT.exists():
    print('RESTORE SAVED STATE:',prior[0])
    with zipfile.ZipFile(prior[0],'r') as z: z.extractall(WORK)
OUT.mkdir(parents=True,exist_ok=True)
print('CORE CONTRACT PASS')
print('DATA:',DATA)
for seed in SEEDS: print('D0',seed,':',D0[seed])
print('OUTPUT:',OUT)

In [ ]:
# Frozen implementation tests + one static audit per seed-matched D0.
subprocess.run([sys.executable,'-m','pytest','-q',
                'tests/test_af2_ffa.py',
                'tests/test_af2_ffa_from_start_dct.py',
                'tests/test_af2_ffa_from_start_decision.py'],cwd=REPO,check=True)
from coffee_detector.af2_ffa import run_af2_ffa_from_start_static_audit
STATIC={}
for seed in SEEDS:
    path=OUT/'static_audits'/f'from_start_static_audit_seed{seed}.json'
    path.parent.mkdir(parents=True,exist_ok=True)
    audit=run_af2_ffa_from_start_static_audit(D0[seed],path,device='cuda:0')
    if audit.get('decision')!='PASS' or audit.get('training_authorized') is not True or audit.get('test_access_authorized') is not False:
        raise RuntimeError(f'STOP: static audit seed {seed} gagal.')
    STATIC[seed]=path
    print('STATIC PASS:',seed,path)

In [ ]:
# Helpers: validate/execute one frozen arm and snapshot completed state.
from coffee_detector.af2_spectral.audit import sha256

def snapshot_state():
    base=str(STATE_ZIP.with_suffix(''))
    if STATE_ZIP.exists(): STATE_ZIP.unlink()
    archive=Path(shutil.make_archive(base,'zip',root_dir=WORK,base_dir=OUT.name))
    print('STATE SNAPSHOT:',archive,archive.stat().st_size,'bytes',flush=True)
    return archive

def validate_result(path,arm,seed):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    if payload.get('format')!='coffee_detector.af2_ffa.from_start_arm_result.v1': raise RuntimeError(f'Format result salah: {path}')
    if payload.get('arm')!=arm or int(payload.get('seed'))!=seed: raise RuntimeError(f'Arm/seed result salah: {path}')
    if payload.get('evaluation_split')!='val' or payload.get('test_images_accessed') is not False: raise RuntimeError(f'Test-lock/result split salah: {path}')
    if payload.get('initial_d0_checkpoint_sha256')!=sha256(D0[seed]): raise RuntimeError(f'D0 provenance salah: {path}')
    ckpt=Path(payload['checkpoint'])
    if not ckpt.is_file(): raise FileNotFoundError(f'Checkpoint result hilang: {ckpt}')
    return payload

def run_one(arm,seed,stage1=None):
    result=OUT/'val_reports'/f'{arm}_seed{seed}_result.json'
    log=OUT/'logs'/f'{arm}_seed{seed}.log'; log.parent.mkdir(parents=True,exist_ok=True)
    if result.is_file():
        payload=validate_result(result,arm,seed)
        print('REUSE COMPLETE:',arm,seed,{k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
        return result
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_from_start_arm',
         '--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),
         '--d0-checkpoint',str(D0[seed]),'--static-audit',str(STATIC[seed]),
         '--output-root',str(OUT),'--seed',str(seed),'--device','0','--authorize-training']
    if stage1 is not None: cmd += ['--stage1-decision',str(stage1)]
    print('\nSTART/RESUME:',arm,'seed',seed,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    shown=-1
    while p.poll() is None:
        csv=OUT/arm/f'{arm}_seed{seed}'/'results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=shown:
            print(f'{arm} seed {seed}: {epochs}/50 epoch tercatat',flush=True); shown=epochs
        time.sleep(120)
    if p.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<no log>'
        raise RuntimeError(f'{arm} seed {seed} gagal, returncode={p.returncode}\n{tail}')
    if not result.is_file(): raise FileNotFoundError(result)
    payload=validate_result(result,arm,seed)
    print('DONE:',arm,seed,{k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
    snapshot_state()
    return result

print('HELPERS READY')

In [ ]:
# ==================== ALL EXPERIMENTS ====================
# Stage 1: AF2FS vs AF2FFAB2FS for all three seeds.
AF2_RESULTS=[]; RFFT_RESULTS=[]
for seed in SEEDS:
    AF2_RESULTS.append(run_one('AF2FS',seed))
    RFFT_RESULTS.append(run_one('AF2FFAB2FS',seed))

STAGE1=OUT/'val_reports'/'af2_ffab2_from_start_decision.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_from_start_decision',
     '--af2',*[str(p) for p in AF2_RESULTS],
     '--ffab2',*[str(p) for p in RFFT_RESULTS],
     '--output',str(STAGE1)]
subprocess.run(cmd,cwd=REPO,check=True)
stage1=json.loads(STAGE1.read_text(encoding='utf-8'))
print('\n================ STAGE 1 DECISION ================')
print('DECISION:',stage1['decision']); print('NEXT:',stage1['next']); print(json.dumps(stage1['aggregate'],indent=2))
snapshot_state()

DCT_RESULTS=[]
DCT_DECISION=None
if stage1['decision']=='PASS' and stage1['next']=='AUTHORIZE_DCT_EFFICIENCY_STAGE':
    print('\nSTAGE 1 PASS -> DCT STAGE AUTHORIZED')
    for seed in SEEDS:
        DCT_RESULTS.append(run_one('AF2FFADCTFS',seed,stage1=STAGE1))

    DCT_DECISION=OUT/'val_reports'/'af2_ffab2_dct_efficiency_decision.json'
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_dct_decision',
         '--rfft',*[str(p) for p in RFFT_RESULTS],
         '--dct',*[str(p) for p in DCT_RESULTS],
         '--output',str(DCT_DECISION),'--device','0']
    subprocess.run(cmd,cwd=REPO,check=True)
    dct=json.loads(DCT_DECISION.read_text(encoding='utf-8'))
    print('\n================ DCT DECISION ================')
    print('DECISION:',dct['decision']); print('NEXT:',dct['next'])
    print('ACCURACY:',json.dumps(dct['aggregate'],indent=2))
    print('EFFICIENCY:',json.dumps(dct['efficiency'],indent=2))
    snapshot_state()
else:
    print('\nSTAGE 1 REJECT -> DCT TIDAK DIJALANKAN. Frozen protocol berhenti di sini.')

In [ ]:
# Final export. Kaggle Saved Version akan menyimpan state ZIP + final output ZIP.
final_manifest={
    'format':'coffee_detector.af2_ffa.all_seeds_all_stages_kaggle.v1',
    'branch':BRANCH,'commit':COMMIT,'seeds':list(SEEDS),
    'stage1_decision':str(STAGE1),
    'stage1':json.loads(STAGE1.read_text(encoding='utf-8')),
    'dct_decision':str(DCT_DECISION) if DCT_DECISION else None,
    'dct_ran':bool(DCT_RESULTS),
    'evaluation_split':'val','test_images_accessed':False,'test_opened':False,
}
MANIFEST=WORK/'af2_ffab2_all_seeds_all_stages_manifest.json'
MANIFEST.write_text(json.dumps(final_manifest,indent=2)+'\n',encoding='utf-8')
snapshot_state()
FINAL_ZIP=Path(shutil.make_archive(str(WORK/'af2-ffab2-all-seeds-all-stages-output'),'zip',root_dir=OUT))
print('\n================ FINAL ================')
print('STAGE1:',final_manifest['stage1']['decision'],final_manifest['stage1']['next'])
if DCT_DECISION:
    dct=json.loads(DCT_DECISION.read_text(encoding='utf-8')); print('DCT:',dct['decision'],dct['next'])
else:
    print('DCT: NOT RUN (Stage 1 did not authorize)')
print('STATE ZIP:',STATE_ZIP)
print('FINAL ZIP:',FINAL_ZIP)
print('MANIFEST:',MANIFEST)
print('TEST: LOCKED / NEVER OPENED')